https://stackoverflow.com/questions/47664061/how-to-apply-polynomial-transformation-to-subset-of-features-in-scikitlearn

In [40]:
import pandas as pd
import numpy as np
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [19]:
X = pd.DataFrame({'cat': ['a', 'b', 'c'], 'num': [1, 2, 3]})
X

,cat,num
0,a,1
1,b,2
2,c,3


In [20]:
theX = pd.get_dummies(X, drop_first=True); theX

,num,cat_b,cat_c
0,1,0,0
1,2,1,0
2,3,0,1


In [21]:
# was class ColumnExtractor(object):
# that was the problem

class ColumnExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_cols = X[self.columns]
        return X_cols

In [22]:
pipeline = Pipeline([
    ('features', FeatureUnion([('num', Pipeline([('extract', ColumnExtractor(columns=['num'])),
                                                 ('poly', PolynomialFeatures(degree=2))
                                                ])),
                               ('cat_var', ColumnExtractor(columns=['cat_b','cat_c']))])
    )])

In [23]:
pipeline.get_params().keys()

dict_keys(['memory', 'steps', 'features', 'features__n_jobs', 'features__transformer_list', 'features__transformer_weights', 'features__num', 'features__cat_var', 'features__num__memory', 'features__num__steps', 'features__num__extract', 'features__num__poly', 'features__num__extract__columns', 'features__num__poly__degree', 'features__num__poly__include_bias', 'features__num__poly__interaction_only', 'features__cat_var__columns'])

In [24]:
pipeline.get_params()['features__num__poly__degree']

2

In [25]:
pipeline.fit_transform(theX)

array([[1., 1., 1., 0., 0.],
       [1., 2., 4., 1., 0.],
       [1., 3., 9., 0., 1.]])

In [26]:
pipeline.set_params(features__num__poly__degree=3)

Pipeline(memory=None,
     steps=[('features', FeatureUnion(n_jobs=1,
       transformer_list=[('num', Pipeline(memory=None,
     steps=[('extract', ColumnExtractor(columns=['num'])), ('poly', PolynomialFeatures(degree=3, include_bias=True, interaction_only=False))])), ('cat_var', ColumnExtractor(columns=['cat_b', 'cat_c']))],
       transformer_weights=None))])

In [27]:
pipeline.get_params()['features__num__poly__degree']

3

In [28]:
pipeline.fit_transform(theX)

array([[ 1.,  1.,  1.,  1.,  0.,  0.],
       [ 1.,  2.,  4.,  8.,  1.,  0.],
       [ 1.,  3.,  9., 27.,  0.,  1.]])

In [46]:
X = pd.DataFrame({'cat': ['a', 'b', 'c'], 'n1': [1, 2, 3], 'n2':[5, 7, 9] })
X

,cat,n1,n2
0,a,1,5
1,b,2,7
2,c,3,9


In [52]:
pipe2nvars = Pipeline([
    ('features', FeatureUnion([('num', 
                                Pipeline([('extract', 
                                           ColumnExtractor(columns=['n1', 'n2'])),
                                          ('poly', 
                                           PolynomialFeatures())  ])),
                               ('cat_var', 
                                ColumnExtractor(columns=['cat_b','cat_c']))])
    )])

In [44]:
np.set_printoptions(precision=3, suppress=True)

In [53]:
for p in range(1, 4):
    pipe2nvars.set_params(features__num__poly__degree=p)
    res = pipe2nvars.fit_transform(pd.get_dummies(X, drop_first=True))
    print('polynomial degree: {}; shape: {}'.format(p, res.shape))
    print(res)

polynomial degree: 1; shape: (3, 5)
[[1. 1. 5. 0. 0.]
 [1. 2. 7. 1. 0.]
 [1. 3. 9. 0. 1.]]
polynomial degree: 2; shape: (3, 8)
[[ 1.  1.  5.  1.  5. 25.  0.  0.]
 [ 1.  2.  7.  4. 14. 49.  1.  0.]
 [ 1.  3.  9.  9. 27. 81.  0.  1.]]
polynomial degree: 3; shape: (3, 12)
[[  1.   1.   5.   1.   5.  25.   1.   5.  25. 125.   0.   0.]
 [  1.   2.   7.   4.  14.  49.   8.  28.  98. 343.   1.   0.]
 [  1.   3.   9.   9.  27.  81.  27.  81. 243. 729.   0.   1.]]


# SO post

In response to the answer from Peng Jun Huang - the approach is terrific but implementation has issues.
(This should be a comment but it's a bit long for that.  Also, don't have enough cookies for that.)

I tried to use the code and had some problems.  After examining I found the following answer to the original question.
The main issue is that the ColumnExtractor needs to inherit from BaseEstimator and TransformerMixin to turn it into an
estimator that can be used with other sklearn tools.

```
import pandas as pd
import numpy as np
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

X = pd.DataFrame({'cat': ['a', 'b', 'c'], 'n1': [1, 2, 3], 'n2':[5, 7, 9] })

   cat	n1	n2
0	a	1	5
1	b	2	7
2	c	3	9

# original version had class ColumnExtractor(object)
# estimators need to inherit from these classes to play nicely with others
class ColumnExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_cols = X[self.columns]
        return X_cols

# Using pandas get dummies to make pipeline a bit simpler avoiding one-hot and label encoder     
# Build the pipeline from a FeatureUnion that processes numerical and one-hot encoded separately.
# FeatureUnion puts them back together when it's done.
pipe2nvars = Pipeline([
    ('features', FeatureUnion([('num', Pipeline([('extract', ColumnExtractor(columns=['n1', 'n2'])),
                                                 ('poly', PolynomialFeatures())
                                                ])),
                               ('cat_var', ColumnExtractor(columns=['cat_b','cat_c']))])
    )])

# now show it working...
for p in range(1, 4):
    pipe2nvars.set_params(features__num__poly__degree=p)
    res = pipe2nvars.fit_transform(pd.get_dummies(X, drop_first=True))
    print('polynomial degree: {}; shape: {}'.format(p, res.shape))
    print(res)
    
polynomial degree: 1; shape: (3, 5)
[[1. 1. 5. 0. 0.]
 [1. 2. 7. 1. 0.]
 [1. 3. 9. 0. 1.]]
polynomial degree: 2; shape: (3, 8)
[[ 1.  1.  5.  1.  5. 25.  0.  0.]
 [ 1.  2.  7.  4. 14. 49.  1.  0.]
 [ 1.  3.  9.  9. 27. 81.  0.  1.]]
polynomial degree: 3; shape: (3, 12)
[[  1.   1.   5.   1.   5.  25.   1.   5.  25. 125.   0.   0.]
 [  1.   2.   7.   4.  14.  49.   8.  28.  98. 343.   1.   0.]
 [  1.   3.   9.   9.  27.  81.  27.  81. 243. 729.   0.   1.]]
```